In [ ]:
# Mount Google Drive to access input/output data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Install required geospatial libraries
# rioxarray : raster I/O and spatial operations built on xarray + rasterio
# cartopy   : map projections and geographic features for visualization
!pip install rioxarray cartopy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 62.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Imports and Configuration
# ============================================================
import os
from os import makedirs
from os.path import join, basename, splitext
from datetime import datetime
from glob import glob
import numpy as np
import rioxarray
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# --- User configuration ---
# Folder containing geolocation-corrected ECOSTRESS LST GeoTIFFs
INPUT_DIR  = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST"

# Folder where all composite GeoTIFFs will be saved
OUTPUT_DIR = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST/Composites"

makedirs(OUTPUT_DIR, exist_ok=True)

# --- Study area bounds (Belize Barrier Reef System) ---
BELIZE_EXTENT = [-89.5, -87.3, 15.7, 18.5]  # [lon_min, lon_max, lat_min, lat_max]

# --- Seasonal groupings for seasonal composites ---
# Each entry maps a label to the list of YYYY-MM strings it spans.
SEASON_GROUPS = {
    "MJJ_2023"     : ["2023-05", "2023-06", "2023-07"],
    "ASO_2023"     : ["2023-08", "2023-09", "2023-10"],
    "NDJ_2023_2024": ["2023-11", "2023-12", "2024-01"],
    "FMA_2024"     : ["2024-02", "2024-03", "2024-04"],
}

# --- Cartopy map features (shared across visualization cells) ---
RESOL = '10m'
LAND_FEATURE = cartopy.feature.NaturalEarthFeature(
    'physical', 'land', scale=RESOL, edgecolor='k', facecolor=cfeature.COLORS['land']
)
BORDER_FEATURE = cartopy.feature.NaturalEarthFeature(
    category='cultural', name='admin_0_boundary_lines_land',
    scale=RESOL, facecolor='none', alpha=0.7
)


In [ ]:
# ============================================================
# Group Input Files by Month-Year
# ============================================================
# Parses the 'doyYYYYDDDHHMMSS' timestamp embedded in each
# ECOSTRESS filename and groups files by their calendar month
# (YYYY-MM). The resulting dict is used by all composite cells.
# ============================================================

ST_masked_filenames = sorted(glob(join(INPUT_DIR, "*_LST_doy*.tif")))

file_groups = {}

for filename in ST_masked_filenames:
    base_name = splitext(basename(filename))[0]
    # Find the 'doyYYYYDDDHHMMSS' token
    for part in base_name.split("_"):
        if part.startswith("doy") and len(part) >= 16:
            timestamp_str = part[3:]  # strip 'doy' prefix
            break
    else:
        continue  # skip files with no valid timestamp

    # Parse year, day-of-year, and time components
    dt = datetime.strptime(
        timestamp_str[0:4] + timestamp_str[4:7] + timestamp_str[7:9] +
        timestamp_str[9:11] + timestamp_str[11:13],
        "%Y%j%H%M%S"
    )
    month_year = dt.strftime("%Y-%m")

    file_groups.setdefault(month_year, []).append(filename)


In [ ]:
# ============================================================
# Monthly Composites — Median and Maximum
# ============================================================
# For each calendar month in file_groups, all scenes are
# reprojected to a common grid (first scene as reference),
# fill values (0) are converted to NaN, and both the pixel-wise
# median and maximum are computed and saved as GeoTIFFs.
# ============================================================

def build_monthly_composite(files, reduction):
    """
    Compute a pixel-wise composite from a list of GeoTIFF files.

    Parameters
    ----------
    files     : list of str  - Paths to input GeoTIFFs for one month.
    reduction : str          - 'median' or 'max'.

    Returns
    -------
    xr.DataArray - Composite DataArray with the same CRS and
                   spatial metadata as the first input file.
    """
    ref = rioxarray.open_rasterio(files[0]).squeeze('band', drop=True)

    aligned = []
    for f in files:
        r   = rioxarray.open_rasterio(f).squeeze('band', drop=True)
        arr = r.rio.reproject_match(ref).data
        aligned.append(np.where(arr == 0, np.nan, arr))

    stacked = np.stack(aligned, axis=0)
    result  = np.nanmedian(stacked, axis=0) if reduction == 'median' else np.nanmax(stacked, axis=0)

    composite      = ref.copy(deep=True)
    composite.data = result
    return composite


for month_year, files in sorted(file_groups.items()):
    for reduction in ('median', 'max'):
        composite  = build_monthly_composite(files, reduction)
        label      = reduction.capitalize()
        out_path   = join(OUTPUT_DIR, f"ECOSTRESS_LST_{label}_{month_year}.tif")
        composite.rio.to_raster(out_path, tiled=True, compress='LZW')


In [ ]:
# ============================================================
# Overall Maximum Composite — Full Study Period
# ============================================================
# Computes the single pixel-wise maximum across all scenes in
# INPUT_DIR, regardless of month. Useful as a 'hottest observed
# temperature' reference layer for the entire study period.
#
# Update TIME_FRAME_LABEL to match your study period.
# ============================================================

TIME_FRAME_LABEL = "2023-2024"

all_files = sorted(glob(join(INPUT_DIR, "*_LST_doy*.tif")))
ref       = rioxarray.open_rasterio(all_files[0]).squeeze('band', drop=True)

aligned = []
for f in all_files:
    with rioxarray.open_rasterio(f).squeeze('band', drop=True) as r:
        arr = r.rio.reproject_match(ref).data
        aligned.append(np.where(arr == 0, np.nan, arr))

max_data      = np.nanmax(np.stack(aligned, axis=0), axis=0)
composite     = ref.copy(deep=True)
composite.data = max_data

out_path = join(OUTPUT_DIR, f"ECOSTRESS_LST_Max_{TIME_FRAME_LABEL}.tif")
composite.rio.to_raster(out_path, tiled=True, compress='LZW')


In [ ]:
# ============================================================
# Seasonal Composites — Median and Maximum
# ============================================================
# Aggregates all scenes within each season defined in
# SEASON_GROUPS and computes both a pixel-wise median and
# maximum composite, saving each as a GeoTIFF.
# ============================================================

for season, months in SEASON_GROUPS.items():
    # Collect all files belonging to this season
    season_files = [f for m in months for f in file_groups.get(m, [])]
    if not season_files:
        continue

    ref = rioxarray.open_rasterio(season_files[0]).squeeze('band', drop=True)

    aligned = []
    for f in season_files:
        r   = rioxarray.open_rasterio(f).squeeze('band', drop=True)
        arr = r.rio.reproject_match(ref).data
        aligned.append(np.where(arr == 0, np.nan, arr))

    stacked = np.stack(aligned, axis=0)

    for reduction, func in [('Median', np.nanmedian), ('Max', np.nanmax)]:
        composite      = ref.copy(deep=True)
        composite.data = func(stacked, axis=0)
        out_path       = join(OUTPUT_DIR, f"ECOSTRESS_{season}_{reduction}_LST.tif")
        composite.rio.to_raster(out_path, tiled=True, compress='LZW')


In [ ]:
# ============================================================
# Visualize Monthly Composites
# ============================================================
# Plots each monthly median and maximum composite as a Cartopy
# map over the Belize study region. A light Gaussian blur
# (sigma=0.5) is applied before plotting to smooth raster
# edges; the underlying saved GeoTIFFs are unaffected.
#
# Adjust VMIN/VMAX to suit your data range.
# ============================================================

VMIN, VMAX = 25, 35

def plot_composite_map(da, title, extent, vmin, vmax, cmap='jet'):
    """
    Plot a 2-D DataArray as a Cartopy map over the study region.

    Parameters
    ----------
    da     : xr.DataArray  - 2-D composite raster.
    title  : str           - Figure title.
    extent : list          - [lon_min, lon_max, lat_min, lat_max].
    vmin   : float         - Colorbar minimum.
    vmax   : float         - Colorbar maximum.
    cmap   : str           - Matplotlib colormap name.
    """
    # Apply light Gaussian smoothing (display only — does not modify saved files)
    smoothed      = gaussian_filter(np.ma.masked_invalid(da.data), sigma=0.5)
    da_plot       = da.copy(deep=True)
    da_plot.data  = smoothed

    fig = plt.figure(figsize=(10, 6))
    ax  = plt.axes(projection=ccrs.PlateCarree())

    im = ax.imshow(da_plot, cmap=cmap, vmin=vmin, vmax=vmax,
                   extent=extent, transform=ccrs.PlateCarree())
    ax.coastlines(resolution='10m', color='black', linewidth=1)
    ax.add_feature(LAND_FEATURE, facecolor='dimgrey', alpha=0.5)
    ax.add_feature(BORDER_FEATURE, linestyle='--', edgecolor='k')
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=14)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Surface Temperature (\u00b0C)")
    plt.tight_layout()
    plt.show()


for reduction_label, pattern in [('Median', 'ECOSTRESS_LST_Median_*.tif'),
                                   ('Max',    'ECOSTRESS_LST_Max_*.tif')]:
    for f in sorted(glob(os.path.join(OUTPUT_DIR, pattern))):
        month = basename(f).split('_')[-1].replace('.tif', '')
        da    = rioxarray.open_rasterio(f).squeeze('band', drop=True)
        if np.isnan(da.data).all():
            continue
        plot_composite_map(da, f"{reduction_label} ECOSTRESS LST — {month}",
                           BELIZE_EXTENT, VMIN, VMAX)


In [ ]:
# ============================================================
# Valid Pixel Count Maps — Per Month
# ============================================================
# For each month, counts the number of scenes that contributed
# a valid (non-NaN, non-zero) observation per pixel. Useful
# as a data-availability layer to assess composite reliability.
# ============================================================

land_gray   = cfeature.NaturalEarthFeature('physical', 'land', scale='10m',
                                            facecolor='dimgrey', edgecolor='k')
borders_gray = cfeature.NaturalEarthFeature('cultural', 'admin_0_boundary_lines_land',
                                             scale='10m', facecolor='none',
                                             edgecolor='black', linestyle='--')

for month, files in sorted(file_groups.items()):
    ref     = rioxarray.open_rasterio(files[0]).squeeze('band', drop=True)
    stacked = []
    for f in files:
        r      = rioxarray.open_rasterio(f).squeeze('band', drop=True)
        r      = r.rio.reproject_match(ref)
        r.data = np.where(r.data == 0, np.nan, r.data)
        stacked.append(r)

    valid_counts = np.sum(~np.isnan(np.stack([r.data for r in stacked])), axis=0)

    fig = plt.figure(figsize=(10, 6))
    ax  = plt.axes(projection=ccrs.PlateCarree())
    im  = ax.imshow(valid_counts, cmap='viridis',
                    extent=BELIZE_EXTENT, transform=ccrs.PlateCarree())
    ax.add_feature(land_gray)
    ax.add_feature(borders_gray)
    ax.coastlines(resolution='10m', color='black', linewidth=1)
    ax.set_extent(BELIZE_EXTENT, crs=ccrs.PlateCarree())
    ax.set_title(f"Valid Pixel Count — {month}", fontsize=14)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Number of Valid Observations")
    plt.tight_layout()
    plt.show()
